# Qwen3-TTS LoRA — обучение голоса на русском
Три шага: установка, Google Drive, запуск интерфейса. Для Colab рекомендуется T4 GPU.


## 1. Установка
Эта ячейка получает свежий код ветки и ставит отдельные окружения для Qwen3-TTS и Qwen3-ASR.


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

if not __import__('torch').cuda.is_available():
    raise RuntimeError('GPU не найден. В Colab выберите среду выполнения с T4 GPU.')
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} ГБ')

CODE = Path('/content/qwen3-tts-lora-code')
REPO = 'https://github.com/egor125552/audio-restoration-colab.git'
BRANCH = 'agent/qwen3-tts-lora-colab'
if CODE.exists():
    shutil.rmtree(CODE)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO,str(CODE)], check=True)
subprocess.run(['bash', str(CODE/'qwen3_tts_lora_colab/install.sh'), '/content/qwen3-tts-trainer'], check=True)
print('Установка завершена.')


## 2. Google Drive
Проекты, checkpoint, готовые LoRA и кэш моделей сохраняются на Google Drive. Служебные файлы Gradio остаются на локальном диске Colab, чтобы публичный туннель мог нормально запускаться.


In [ ]:
import os
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Qwen3-TTS Training')
ROOT.mkdir(parents=True, exist_ok=True)
MODEL_CACHE = ROOT / '.cache' / 'huggingface' / 'hub'
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
LOCAL_HF_HOME = Path('/content/qwen3-hf-home')
LOCAL_HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ['QWEN_TRAIN_DRIVE_ROOT'] = str(ROOT)
os.environ['HF_HOME'] = str(LOCAL_HF_HOME)
os.environ['HF_HUB_CACHE'] = str(MODEL_CACHE)
print(f'Проекты и checkpoint: {ROOT}')
print(f'Кэш моделей: {MODEL_CACHE}')
print(f'Локальные служебные файлы Hugging Face/Gradio: {LOCAL_HF_HOME}')


## 3. Запуск интерфейса
Эта ячейка **должна работать постоянно**. Это не зависание. Gradio создаст публичную ссылку `gradio.live` и напечатает её ниже. Если туннель не создастся, будет показана настоящая причина ошибки вместо ложного сообщения об успешном запуске. Вывод нарезки, ASR и обучения остаётся видимым ниже; полоски прогресса перерисовываются на месте. Чтобы остановить сервер, прервите выполнение ячейки.


In [ ]:
import os, sys
from pathlib import Path

CODE = Path('/content/qwen3-tts-lora-code')
sys.path.insert(0, str(CODE / 'qwen3_tts_lora_colab'))
from colab_stream import run_streamed

os.environ['PYTHONUNBUFFERED'] = '1'
cmd = [
    '/content/qwen3-tts-trainer/tts-env/bin/python', '-u',
    str(CODE / 'qwen3_tts_lora_colab/launch_colab.py')
]

print('Запускаю интерфейс. Публичная ссылка Gradio появится ниже.', flush=True)
code = run_streamed(cmd)
if code != 0:
    raise RuntimeError(f'Интерфейс завершился с кодом {code}')
